# Stage 2 Prototype — Competitive Intelligence on Linear & Asana

**One question to answer:** Can Claude actually produce useful, non-generic competitive intelligence from real public data on a single competitor?

**If yes →** proceed to Stage 3 (production agent).
**If no →** revisit prompt strategy, data sources, or the workflow choice itself.

## What this notebook does

1. Fetches real public pages for Linear and Asana (pricing, changelog/blog)
2. Strips HTML to text
3. Sends the cleaned text to Claude with a signal-extraction prompt
4. Returns structured signals (Pydantic schema)
5. Renders an intelligence report

## Honesty rules

- **No mock data.** Real URLs only.
- **Verifiable signals.** Every signal must include a source URL and a verbatim quote.
- **Manual judgment at the end.** I have to look at the output and ask: *would I forward this to a CMO?*

Run prerequisites: `pip install -r ../requirements.txt` and set `ANTHROPIC_API_KEY` in `../.env`.

## Setup

In [ ]:
import os
import json
import time
from pathlib import Path

import httpx
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from anthropic import Anthropic

# override=True is critical: some shells pre-set ANTHROPIC_API_KEY to an empty
# string, which python-dotenv refuses to overwrite by default.
load_dotenv(Path('..') / '.env', override=True)

API_KEY = os.environ.get('ANTHROPIC_API_KEY')
if not API_KEY:
    raise EnvironmentError('ANTHROPIC_API_KEY is not set. Copy .env.example to .env and fill it in.')

client = Anthropic(api_key=API_KEY)

# Prototype uses Sonnet 4.6 for cheap iteration. Production will A/B test Opus 4.7 vs Sonnet 4.6 on signal quality.
MODEL_ID = 'claude-sonnet-4-6'
MAX_BYTES_PER_FETCH = 5 * 1024 * 1024  # 5 MB cap (Linear pages are ~2MB raw HTML)
REQUEST_TIMEOUT_SECONDS = 15.0
USER_AGENT = 'Mozilla/5.0 (compatible; B2BAgent-Prototype/0.1; +contact@example.com)'

print(f'Using model: {MODEL_ID}')
print(f'API key loaded: {API_KEY[:10]}... ({len(API_KEY)} chars)')

## Step 1 — Define competitor sources

Two competitors, two sources each. We're deliberately starting narrow: pricing pages (catch pricing/plan changes) and changelog/blog feeds (catch product launches). Stage 3 will add careers pages, GitHub, RSS, etc.

In [31]:
COMPETITORS = {
    'Linear': [
        {'kind': 'pricing', 'url': 'https://linear.app/pricing'},
        {'kind': 'changelog', 'url': 'https://linear.app/changelog'},
    ],
    'Asana': [
        {'kind': 'pricing', 'url': 'https://asana.com/pricing'},
        {'kind': 'blog_rss', 'url': 'https://blog.asana.com/feed/'},
    ],
}

for name, sources in COMPETITORS.items():
    print(f'{name}: {len(sources)} source(s)')
    for s in sources:
        print(f'  - {s["kind"]:10}  {s["url"]}')

Linear: 2 source(s)
  - pricing     https://linear.app/pricing
  - changelog   https://linear.app/changelog
Asana: 2 source(s)
  - pricing     https://asana.com/pricing
  - blog_rss    https://blog.asana.com/feed/


## Step 2 — Fetch with security guards

Per global security rules (`~/.claude/rules/security.md`): explicit timeout, response size cap via streaming, no automatic redirect following (each redirect target re-validated). For prototype scope we're hitting hardcoded competitor URLs (allowlist), so the SSRF DNS-resolution check is overkill — but `fetch_with_guards()` already enforces timeout + size cap so the production code can reuse this shape.

In [ ]:
ALLOWED_HOSTS = {
    'linear.app', 'asana.com', 'blog.asana.com',
    'www.notion.so', 'monday.com', 'clickup.com',
}

def fetch_with_guards(url: str) -> str:
    """Fetch a URL with timeout, size cap, and host allowlist. Returns raw response body."""
    parsed_host = httpx.URL(url).host
    if parsed_host not in ALLOWED_HOSTS:
        raise ValueError(f'Host {parsed_host!r} not in allowlist')

    with httpx.Client(timeout=REQUEST_TIMEOUT_SECONDS, follow_redirects=False, headers={'User-Agent': USER_AGENT}) as http:
        # Follow up to 3 redirects manually, re-checking host each time
        current_url = url
        for _ in range(4):
            with http.stream('GET', current_url) as response:
                if response.status_code in (301, 302, 307, 308):
                    raw_location = response.headers.get('location', '')
                    if not raw_location:
                        raise RuntimeError(f'Redirect with no location header at {current_url}')
                    # Resolve relative URLs (e.g., "/path/feed/") against the current URL.
                    # Without this, Asana's edge redirect to "/blog-edge-redirect/feed/" fails.
                    next_url = str(httpx.URL(current_url).join(raw_location))
                    next_host = httpx.URL(next_url).host
                    if next_host and next_host not in ALLOWED_HOSTS:
                        raise ValueError(f'Redirect target host {next_host!r} not in allowlist')
                    current_url = next_url
                    continue
                response.raise_for_status()
                chunks, received = [], 0
                for chunk in response.iter_bytes(chunk_size=65536):
                    chunks.append(chunk)
                    received += len(chunk)
                    if received >= MAX_BYTES_PER_FETCH:
                        break
                return b''.join(chunks).decode('utf-8', errors='replace')
        raise RuntimeError(f'Too many redirects starting from {url}')


def html_to_text(html: str) -> str:
    """Strip HTML to readable text. Drops script/style/nav noise."""
    soup = BeautifulSoup(html, 'lxml')
    for tag in soup(['script', 'style', 'noscript', 'svg', 'link', 'meta']):
        tag.decompose()
    text = soup.get_text(separator='\n', strip=True)
    # Collapse runs of blank lines
    lines = [ln for ln in (l.strip() for l in text.splitlines()) if ln]
    return '\n'.join(lines)

In [33]:
# Fetch all sources. Be polite: 1s between requests.
fetched = {}
for competitor, sources in COMPETITORS.items():
    fetched[competitor] = []
    for source in sources:
        try:
            raw = fetch_with_guards(source['url'])
            cleaned = html_to_text(raw)
            fetched[competitor].append({
                'kind': source['kind'],
                'url': source['url'],
                'raw_bytes': len(raw),
                'text_chars': len(cleaned),
                'text': cleaned,
            })
            print(f'  {competitor:8} {source["kind"]:10} {len(raw):>10}b raw -> {len(cleaned):>7} chars text')
        except Exception as exc:
            print(f'  {competitor:8} {source["kind"]:10} FAILED: {exc}')
        time.sleep(1.0)

  Linear   pricing       1855490b raw ->    2889 chars text
  Linear   changelog     2010819b raw ->   55404 chars text
  Asana    pricing       1009727b raw ->   40218 chars text
  Asana    blog_rss   FAILED: unknown url type: '/blog-edge-redirect/feed/'


## Step 3 — The extraction prompt

**Design decisions worth noting:**

1. **Strict output contract via tool use.** We force Claude to return JSON matching a Pydantic schema. Malformed output fails loudly, not silently.
2. **Every signal requires a verbatim quote.** This is the anti-hallucination guard. If Claude can't quote the source, the signal doesn't exist.
3. **Signal types are constrained.** Five buckets (`product_launch`, `pricing_change`, `hiring`, `positioning`, `content`). No `"other"` escape hatch — forces the model to commit.
4. **Significance score 1–5.** Required, with a rubric. Forces the model to rank rather than dump everything as 'interesting'.

In [34]:
class Signal(BaseModel):
    signal_type: str = Field(description='One of: product_launch, pricing_change, hiring, positioning, content')
    headline: str = Field(description='One-sentence summary suitable for an email subject line')
    detail: str = Field(description='2-3 sentence explanation of what was observed and why it matters')
    verbatim_quote: str = Field(description='Direct quote from the source proving the signal is real')
    source_url: str = Field(description='URL where the signal was observed')
    significance: int = Field(ge=1, le=5, description='1=trivia, 3=worth knowing, 5=strategic')

class IntelligenceReport(BaseModel):
    competitor: str
    signals: list[Signal]
    null_finding: str | None = Field(default=None, description='If no signals worth surfacing, explain why')

EXTRACT_TOOL = {
    'name': 'submit_intelligence_report',
    'description': 'Submit the competitive intelligence report after analyzing the fetched data.',
    'input_schema': IntelligenceReport.model_json_schema(),
}

SYSTEM_PROMPT = '''You are a senior B2B SaaS competitive intelligence analyst. You are reviewing fetched public data about a competitor to produce a daily intelligence digest for a marketing team.

Your job is to surface signals that would make a CMO say "I didn't know that, and it changes how I'm thinking." Skip generic observations.

Rules you must follow:
1. Every signal must be backed by a verbatim quote from the source data. If you cannot quote it, do not report it.
2. Significance ratings: 1 = trivia (skip unless slow week), 3 = worth knowing (informs positioning), 5 = strategic (changes battle cards, sales scripts, or roadmap discussions).
3. Prefer 0-3 high-signal items over 10 generic ones. Use `null_finding` if there is genuinely nothing of strategic interest.
4. Signal types: product_launch (new feature shipped), pricing_change (plan/price modified), hiring (key role posted that signals strategy), positioning (new messaging/category claim), content (notable new thought-leadership piece).
5. Be specific. "Linear added a new feature" is useless. "Linear shipped Cycles 2.0 with auto-rollover; targets engineering managers; quote: ..." is useful.

Submit your report via the submit_intelligence_report tool.'''

print('Prompt and schema configured.')

Prompt and schema configured.


## Step 4 — Call Claude per competitor

In [35]:
def extract_signals(competitor: str, sources: list[dict]) -> tuple[IntelligenceReport, dict]:
    """Call Claude with the fetched sources, return the structured report + usage info."""
    source_blocks = []
    for s in sources:
        truncated = s['text'][:30_000]  # ~7.5k tokens per source max
        truncation_note = ' [TRUNCATED]' if len(s['text']) > 30_000 else ''
        source_blocks.append(
            f"=== Source: {s['kind']} ===\n"
            f"URL: {s['url']}{truncation_note}\n\n"
            f"{truncated}"
        )

    user_message = (
        f"Competitor: {competitor}\n"
        f"Number of sources: {len(sources)}\n\n"
        + '\n\n'.join(source_blocks)
    )

    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=4096,
        system=SYSTEM_PROMPT,
        tools=[EXTRACT_TOOL],
        tool_choice={'type': 'tool', 'name': 'submit_intelligence_report'},
        messages=[{'role': 'user', 'content': user_message}],
    )

    tool_use = next((b for b in response.content if b.type == 'tool_use'), None)
    if not tool_use:
        raise RuntimeError(f'No tool_use in response: {response.content}')

    report = IntelligenceReport.model_validate(tool_use.input)
    usage = {
        'input_tokens': response.usage.input_tokens,
        'output_tokens': response.usage.output_tokens,
    }
    return report, usage

In [36]:
reports = {}
total_input_tokens = 0
total_output_tokens = 0

for competitor, sources in fetched.items():
    if not sources:
        print(f'No data for {competitor}, skipping')
        continue
    print(f'\n=== Analyzing {competitor} ===')
    report, usage = extract_signals(competitor, sources)
    reports[competitor] = report
    total_input_tokens += usage['input_tokens']
    total_output_tokens += usage['output_tokens']
    print(f'Returned {len(report.signals)} signal(s) | tokens in/out: {usage["input_tokens"]}/{usage["output_tokens"]}')

# Rough Sonnet 4.6 pricing (verify against current Anthropic pricing page): $3/MTok in, $15/MTok out
est_cost = (total_input_tokens / 1_000_000) * 3.0 + (total_output_tokens / 1_000_000) * 15.0
print(f'\nTotal: {total_input_tokens} in / {total_output_tokens} out  ~= ${est_cost:.3f} this run')


=== Analyzing Linear ===
Returned 6 signal(s) | tokens in/out: 8799/1348

=== Analyzing Asana ===
Returned 4 signal(s) | tokens in/out: 7951/1057

Total: 16750 in / 2405 out  ~= $0.086 this run


## Step 5 — Render the intelligence report

In [37]:
def render_report(report: IntelligenceReport) -> str:
    lines = [f'# {report.competitor}', '']
    if not report.signals:
        lines.append(f'*No signals worth surfacing. Reason:* {report.null_finding or "(none provided)"}')
        return '\n'.join(lines)
    ranked = sorted(report.signals, key=lambda s: -s.significance)
    for s in ranked:
        lines.extend([
            f'## [{s.significance}/5 - {s.signal_type}] {s.headline}',
            '',
            s.detail,
            '',
            f'> {s.verbatim_quote}',
            '',
            f'Source: {s.source_url}',
            '',
        ])
    return '\n'.join(lines)

for competitor, report in reports.items():
    print('=' * 60)
    print(render_report(report))
    print()

# Linear

## [5/5 - product_launch] Linear ships Code Intelligence (beta): AI agent gets read access to your codebase, enabling non-engineers to self-serve technical answers

Code Intelligence gives Linear Agent the ability to reason over connected GitHub repositories, letting PMs, Support, and Sales query how features work without pulling in an engineer. It is available in public beta on Business and Enterprise plans at no extra cost during beta — a deliberate land-and-expand move that lowers the barrier for Business-tier adoption. This directly competes with tools like Notion AI and Confluence Intelligence for the 'shared product context' use case.

> PMs can write sharper specs, Support and Sales can answer technical questions with more confidence, and Engineering can investigate bugs, regressions, and unfamiliar parts of the system faster.

Source: https://linear.app/changelog

## [5/5 - product_launch] Linear launches native Releases feature with CI/CD integration, auto-updating i

## Step 6 — Save the run for the learnings doc

Persist the raw output so we can paste real samples into `PROTOTYPE_LEARNINGS.md` and compare runs as we iterate the prompt.

In [38]:
from datetime import datetime

run_dir = Path('runs')
run_dir.mkdir(exist_ok=True)
stamp = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
out_path = run_dir / f'{stamp}.json'

out_path.write_text(json.dumps(
    {
        'model': MODEL_ID,
        'tokens_in': total_input_tokens,
        'tokens_out': total_output_tokens,
        'est_cost_usd': round(est_cost, 4),
        'reports': {name: rep.model_dump() for name, rep in reports.items()},
    },
    indent=2,
))
print(f'Saved {out_path}')

Saved runs/20260518T192638Z.json


/var/folders/fb/8b7pz749393gl9lh5ppnst7r0000gn/T/ipykernel_30616/4213704124.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  stamp = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')


## Self-evaluation — answer the Stage 2 question

Look at the rendered reports above and answer honestly:

1. **Would you forward this to a CMO?** (yes / no / with edits)
2. **Are the verbatim quotes actually verbatim?** (spot-check 2-3 against the source URLs)
3. **Are the significance ratings calibrated?** (any 5s that should be 2s?)
4. **What's the worst signal in the digest?** (this tells you what to fix first)
5. **What's missing that you'd expect to see?**

Write the answers into `PROTOTYPE_LEARNINGS.md` and move to Stage 3 if the digest passes the "forward to CMO" bar.